In [4]:
import numpy as np
import pandas as pd
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi  # Ensure to install with `pip install rank-bm25`
import torch
from transformers import BertTokenizer, BertModel

# Load metadata from the CSV file
metadata = pd.read_csv('metadata.csv')

# Load the FAISS index
faiss_index = faiss.read_index('bert_avg_document_embeddings.index')

# Check the columns in the metadata to confirm
print(metadata.columns)

# Combine title, snippet, and paragraphs into one searchable field (for sparse search)
# Here we assume your metadata does not have 'Title', 'Snippet', 'Paragraphs' directly,
# but includes fields like 'query', 'rank', and 'keyword' as per your previous code.
# If 'Title', 'Snippet', and 'Paragraphs' were part of `metadata`, add them back.
metadata['combined_text'] = metadata['query']  # Use query or relevant fields for combined text

# Initialize TF-IDF vectorizer and BM25 for keyword matching
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf_vectorizer.fit_transform(metadata['combined_text'])
tokenized_corpus = [doc.split(" ") for doc in metadata['combined_text']]
bm25 = BM25Okapi(tokenized_corpus)

# Load BERT model and tokenizer (for dense search)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Function to generate BERT embeddings (for dense retrieval)
def generate_bert_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # (1, seq_len, 768)
        mask = inputs['attention_mask'].unsqueeze(-1)  # (1, seq_len, 1)
        masked_embeddings = last_hidden * mask  # Zero out padding
        summed = masked_embeddings.sum(dim=1)
        counts = mask.sum(dim=1)
        mean_pooled = summed / counts  # Mean over non-padded tokens
    return mean_pooled.cpu().numpy().flatten()

# Sparse retrieval function (TF-IDF + BM25)
def sparse_retrieval(query, top_k=5):
    query_tfidf = tfidf_vectorizer.transform([query])
    tfidf_scores = np.dot(query_tfidf, tfidf_matrix.T).toarray().flatten()
    tokenized_query = query.split(" ")
    bm25_scores = bm25.get_scores(tokenized_query)
    combined_scores = 0.5 * tfidf_scores + 0.5 * bm25_scores
    top_indices = np.argsort(combined_scores)[::-1][:top_k]
    return top_indices

# Dense retrieval function (FAISS)
def dense_retrieval(query, top_k=5):
    query_embedding = generate_bert_embedding(query)
    query_embedding = np.expand_dims(query_embedding, axis=0).astype('float32')
    distances, indices = faiss_index.search(query_embedding, top_k)
    return indices.flatten()

# Hybrid retrieval (sparse + dense)
def hybrid_retrieval(query, top_k=5):
    sparse_results = sparse_retrieval(query, top_k)
    dense_results = dense_retrieval(query, top_k)
    
    # Combine results with priority to dense
    combined_indices = np.concatenate([dense_results, sparse_results])
    unique_indices = np.unique(combined_indices)

    # Get the top k results prioritizing dense results first
    final_indices = list(set(dense_results) | set(unique_indices))[:top_k]

    # Retrieve metadata for the top results
    results = metadata.iloc[final_indices].to_dict(orient='records')
    return final_indices, results

Index(['query', 'rank', 'type', 'keyword', 'title'], dtype='object')


In [5]:
# Example query
query = "Dimensionality Reduction"
indices, results = hybrid_retrieval(query)

# Output results with indices
for idx, res in zip(indices, results):
    print(f"Index: {idx} -> Result: {res}")

Index: 2 -> Result: {'query': 'Artificial intelligence', 'rank': 3.0, 'type': 'article', 'keyword': 'Artificial intelligence', 'title': 'What Is Artificial Intelligence (AI)? | ...', 'combined_text': 'Artificial intelligence'}
Index: 67 -> Result: {'query': 'analysis', 'rank': 1.0, 'type': 'article', 'keyword': 'analysis', 'title': 'Analysis Definition & Meaning', 'combined_text': 'analysis'}
Index: 70 -> Result: {'query': 'analysis', 'rank': 4.0, 'type': 'article', 'keyword': 'analysis', 'title': 'ANALYSIS Definition & Meaning', 'combined_text': 'analysis'}
Index: 39 -> Result: {'query': 'deep learning', 'rank': 2.0, 'type': 'article', 'keyword': 'deep learning', 'title': 'What Is Deep Learning?', 'combined_text': 'deep learning'}
Index: 85 -> Result: {'query': 'Chapter Title', 'rank': nan, 'type': 'textbook', 'keyword': 'Chapter Keywords', 'title': 'Chapter Title', 'combined_text': 'Chapter Title'}


In [6]:
# Example query
query = "AI"
indices, results = hybrid_retrieval(query)

# Output results with indices
for idx, res in zip(indices, results):
    print(f"Index: {idx} -> Result: {res}")

Index: 3 -> Result: {'query': 'Artificial intelligence', 'rank': 4.0, 'type': 'article', 'keyword': 'Artificial intelligence', 'title': 'Artificial intelligence (AI) | Definition, Examples, Types, ...', 'combined_text': 'Artificial intelligence'}
Index: 67 -> Result: {'query': 'analysis', 'rank': 1.0, 'type': 'article', 'keyword': 'analysis', 'title': 'Analysis Definition & Meaning', 'combined_text': 'analysis'}
Index: 70 -> Result: {'query': 'analysis', 'rank': 4.0, 'type': 'article', 'keyword': 'analysis', 'title': 'ANALYSIS Definition & Meaning', 'combined_text': 'analysis'}
Index: 15 -> Result: {'query': 'Classification', 'rank': 6.0, 'type': 'article', 'keyword': 'Classification', 'title': 'CLASSIFICATION Definition & Meaning', 'combined_text': 'Classification'}
Index: 23 -> Result: {'query': 'Clustering', 'rank': 5.0, 'type': 'article', 'keyword': 'Clustering', 'title': 'What is clustering?', 'combined_text': 'Clustering'}
